# MiniMind 学习笔记

> 参考 [MiniMind 官方文档](https://github.com/jingyaogong/minimind)

本 Notebook 包含两个部分：
1. **测试已有模型效果** — 下载预训练好的模型，直接体验推理
2. **从0开始训练** — 下载数据集，完整复现 Pretrain + SFT 全流程

**环境要求**：Google Colab（GPU 运行时，推荐 A100）

## 环境准备

In [2]:
# 检查 GPU 是否可用
!nvidia-smi

In [3]:
# 挂载 Google Drive (Colab 断开后本地文件会丢失，挂载 Drive 可持久保存模型和数据)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [4]:
# 克隆官方仓库
!git clone https://github.com/jingyaogong/minimind.git

# 进入项目目录
%cd minimind

# 安装所需依赖
!pip install -r requirements.txt

---
## Ⅰ 测试已有模型效果

### 1. 下载预训练模型

将 HuggingFace 上的 MiniMind2 (104M, Transformers 格式) 模型克隆到项目根目录。

In [5]:
# 下载模型到项目根目录
# 国内环境可替换为: !git clone https://www.modelscope.cn/models/gongjy/MiniMind2
!git clone https://huggingface.co/jingyaogong/MiniMind2

In [6]:
# 检查代码和模型是否下载成功
import os

print('=== 检查项目核心文件 ===')
!ls eval_llm.py model/model_minimind.py

print('\n=== 检查模型文件 ===')
!ls -lh MiniMind2/

# 校验关键文件是否存在
config_ok = os.path.exists('MiniMind2/config.json')
weights_ok = os.path.exists('MiniMind2/model.safetensors') or os.path.exists('MiniMind2/pytorch_model.bin')
all_ok = config_ok and weights_ok
print(f'\n✅ 模型文件完整，可以继续！' if all_ok else f'\n❌ 模型文件缺失，请重新运行上方下载单元格！')

### 2. 命令行问答

使用 Transformers 格式模型进行推理（自动测试模式，回答预置的一组问题）。

In [7]:
# 自动测试模式 (echo "0" 自动选择 [0]自动测试)
!echo "0" | python eval_llm.py --load_from MiniMind2

---
## Ⅱ 从0开始训练

完整复现 MiniMind 模型的训练过程：
1. 下载数据集
2. **预训练 (Pretrain)** — 让模型学会「接龙」，积累知识
3. **监督微调 (SFT)** — 让模型学会「对话」，遵循指令
4. 测试自己训练的模型

本 Notebook 训练两个模型规格：
- **MiniMind2-Small (26M)**: hidden_size=512, 8层, 预训练约30分钟
- **MiniMind2 (104M)**: hidden_size=768, 16层, 预训练约2小时

> 使用 `pretrain_hq.jsonl` + `sft_mini_512.jsonl` 最快速度复现。A100-40GB 原生支持 bfloat16，无需 float16 降级。

### 1. 下载数据集

从 HuggingFace 下载最小推荐数据集：
- `pretrain_hq.jsonl` (~1.6GB) — 预训练语料
- `sft_mini_512.jsonl` (~1.2GB) — SFT 对话数据

In [8]:
# 下载数据集 (从 HuggingFace)
# 国内环境可替换为 ModelScope:
#   !pip install modelscope -q
#   !modelscope download --dataset gongjy/minimind_dataset pretrain_hq.jsonl --local_dir ./dataset
#   !modelscope download --dataset gongjy/minimind_dataset sft_mini_512.jsonl --local_dir ./dataset

!mkdir -p dataset
!wget -c -O dataset/pretrain_hq.jsonl "https://huggingface.co/datasets/jingyaogong/minimind_dataset/resolve/main/pretrain_hq.jsonl"
!wget -c -O dataset/sft_mini_512.jsonl "https://huggingface.co/datasets/jingyaogong/minimind_dataset/resolve/main/sft_mini_512.jsonl"

In [9]:
# 检查数据集是否下载成功
import os

print('=== 检查数据集文件 ===')
!ls -lh dataset/

required_datasets = ['dataset/pretrain_hq.jsonl', 'dataset/sft_mini_512.jsonl']
all_ok = all(os.path.exists(f) and os.path.getsize(f) > 1024 for f in required_datasets)
print(f'\n✅ 数据集完整，可以开始训练！' if all_ok else f'\n❌ 数据集缺失或不完整，请重新运行上方下载单元格！')

### 2. 预训练 (Pretrain) — 让模型学会「说话」

预训练让模型从大量文本中学习知识和语言规律。目标只有一个：**学会词语接龙**。

分两阶段训练：
- **阶段 1**: MiniMind2-Small (26M, hidden_size=512, 8层)
- **阶段 2**: MiniMind2 (104M, hidden_size=768, 16层)

A100 优化：使用 `bfloat16` 混合精度（A100 原生支持），更大 batch_size。

> 训练完成后，权重分别保存为 `out/pretrain_512.pth` 和 `out/pretrain_768.pth`。

In [10]:
%cd /content/minimind/trainer

# === 阶段 1: 预训练 MiniMind2-Small (26M, hidden_size=512) ===
# A100 优化: bfloat16 + batch_size=64 + accumulation_steps=4
!python train_pretrain.py \
    --hidden_size 512 \
    --num_hidden_layers 8 \
    --batch_size 64 \
    --accumulation_steps 4 \
    --num_workers 4 \
    --dtype bfloat16

In [11]:
# === 阶段 2: 预训练 MiniMind2 (104M, hidden_size=768) ===
# A100-40GB 可轻松容纳 104M 模型的完整训练
!python train_pretrain.py \
    --hidden_size 768 \
    --num_hidden_layers 16 \
    --batch_size 32 \
    --accumulation_steps 8 \
    --num_workers 4 \
    --dtype bfloat16

### 3. 监督微调 (SFT) — 让模型学会「回答问题」

SFT 阶段在预训练权重的基础上，用对话数据进行微调，让模型学会遵循指令进行对话。

分两阶段微调（对应两个预训练模型）：
- **Small 模型**: 加载 `pretrain_512.pth`，输出 `full_sft_512.pth`
- **Full 模型**: 加载 `pretrain_768.pth`，输出 `full_sft_768.pth`

> 脚本默认加载同 `hidden_size` 的预训练权重 (`--from_weight pretrain`)。

In [ ]:
%cd /content/minimind/trainer

# === 阶段 1: SFT MiniMind2-Small (26M) ===
!python train_full_sft.py \
    --hidden_size 512 \
    --num_hidden_layers 8 \
    --batch_size 64 \
    --accumulation_steps 4 \
    --num_workers 4 \
    --dtype bfloat16

In [ ]:
# === 阶段 2: SFT MiniMind2 (104M) ===
!python train_full_sft.py \
    --hidden_size 768 \
    --num_hidden_layers 16 \
    --batch_size 32 \
    --accumulation_steps 8 \
    --num_workers 4 \
    --dtype bfloat16

### 4. 测试自己训练的模型

分别测试两个模型的 SFT 效果。

In [ ]:
%cd /content/minimind

# 测试 MiniMind2-Small (26M)
!echo "0" | python eval_llm.py --weight full_sft --hidden_size 512 --num_hidden_layers 8

In [ ]:
# 测试 MiniMind2 (104M)
!echo "0" | python eval_llm.py --weight full_sft --hidden_size 768 --num_hidden_layers 16